# Objective 1: Predicting HANNA Scores from Goodreads Reviews

--------------------------------------------------------------------------------------------------------------

**Pipeline overview:**
1. Load & filter the UCSD Goodreads Fantasy & Paranormal reviews
2. Sentence tokenise each review (NLTK)
3. Manual annotation interface (Label Studio export format) & inter-annotator agreement
4. Fine-tune DeBERTa-v3-base as a multi-label sentence classifier across HANNA dimensions
5. Run inference → aggregate per-review proportion scores
6. LLM HANNA scoring (GPT-4o few-shot)
7. Validate: correlate proportion scores with LLM HANNA scores

**HANNA dimensions used (from the HANNA dataset):**
- Coherence
- Empathy
- Surprise
- Engagement
- Complexity
- Relevance

**Mapping to sentence-level annotation dimensions:**
| HANNA | Sentence Dimension |
|---|---|
| Coherence | Narrative Structure & Quality |
| Empathy | Character & Emotion |
| Surprise | Originality |
| Engagement | Immersion + Character & Emotion |
| Complexity | Thematic Depth + Writing Style |
| Relevance | Narrative Structure & Quality + Thematic Depth |

## Import

In [2]:
import os
import json
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from label_studio_sdk import Client
#from langdetect import detect
#import langid
from langid import classify
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
import sentencepiece

## Loading Data

In [3]:
books_file = "/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/raw/goodreads/goodreads_books_fantasy_paranormal.json"
reviews_file = "/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/raw/goodreads/goodreads_reviews_fantasy_paranormal.json"

books = []
with open(books_file, "r") as f:
    for line in tqdm(f):
        books.append(json.loads(line))
book_df = pd.DataFrame(books)
print(book_df.shape)
book_df.head()


reviews = []
with open(reviews_file, "r") as f:
    for line in tqdm(f):
        reviews.append(json.loads(line))
review_df = pd.DataFrame(reviews)
print(review_df.shape)
review_df.head()

merged = review_df.merge(book_df, on='book_id', how='inner')

0it [00:00, ?it/s]

(258585, 29)


0it [00:00, ?it/s]

(3424641, 11)


In [3]:
merged.columns

Index(['user_id', 'book_id', 'review_id', 'rating', 'review_text',
       'date_added', 'date_updated', 'read_at', 'started_at', 'n_votes',
       'n_comments', 'isbn', 'text_reviews_count', 'series', 'country_code',
       'language_code', 'popular_shelves', 'asin', 'is_ebook',
       'average_rating', 'kindle_asin', 'similar_books', 'description',
       'format', 'link', 'authors', 'publisher', 'num_pages',
       'publication_day', 'isbn13', 'publication_month', 'edition_information',
       'publication_year', 'url', 'image_url', 'ratings_count', 'work_id',
       'title', 'title_without_series'],
      dtype='str')

## Cleaning and Filtering

In [4]:
def basic_filter(df: pd.DataFrame,min_words: int = 30, min_rating: int = 1, max_rating: int = 5) -> pd.DataFrame:
    """ Remove empty / very short reviews and invalid ratings. """
    english_codes = ["eng", "en-UK", "en-US", "en-AUS"]

    df = df[df["language_code"].isin(english_codes)]
    df = df.dropna(subset=["review_text", "rating"])
    df = df[df["review_text"].str.split().str.len() >= min_words]
    df = df[df["rating"].between(min_rating, max_rating)]
    df = df.drop_duplicates(subset=["review_text"])
    df = df.reset_index(drop=True)
    return df

df_raw = merged
df = basic_filter(df_raw)
print(f"Loaded {len(df_raw):,} rows → {len(df):,} after filtering")

Loaded 3,424,641 rows → 1,903,442 after filtering


In [20]:
df.columns

Index(['user_id', 'book_id', 'review_id', 'rating', 'review_text',
       'date_added', 'date_updated', 'read_at', 'started_at', 'n_votes',
       'n_comments', 'isbn', 'text_reviews_count', 'series', 'country_code',
       'language_code', 'popular_shelves', 'asin', 'is_ebook',
       'average_rating', 'kindle_asin', 'similar_books', 'description',
       'format', 'link', 'authors', 'publisher', 'num_pages',
       'publication_day', 'isbn13', 'publication_month', 'edition_information',
       'publication_year', 'url', 'image_url', 'ratings_count', 'work_id',
       'title', 'title_without_series'],
      dtype='str')

In [ ]:
#df.to_csv("/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/processed/cleaned_reviews.csv", index=False)
#print(f"Saved cleaned dataset")

## Configuration

In [5]:
OUT_DIR = "outputs"
os.makedirs("data",   exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)
os.makedirs("models", exist_ok=True)

# Model
BASE_MODEL   = "microsoft/deberta-v3-base"   # change to roberta-base if preferred
CLASSIFIER_SAVE_PATH = "models/deberta_hanna_classifier"

# HANNA & My dimensions
SENTENCE_DIMS = ["Narrative Structure & Quality", "Character & Emotion","Originality", "Immersion", "Thematic Depth", "Writing Style",]
HANNA_DIMS = ["Coherence", "Empathy", "Surprise", "Engagement", "Complexity", "Relevance"]

# Mapping: HANNA dimension to which sentence dims contribute
HANNA_MAPPING = {
    "Coherence":   ["Narrative Structure & Quality"],
    "Empathy":     ["Character & Emotion"],
    "Surprise":    ["Originality"],
    "Engagement":  ["Immersion", "Character & Emotion"],
    "Complexity":  ["Thematic Depth", "Writing Style"],
    "Relevance":   ["Narrative Structure & Quality", "Thematic Depth"],
}

# Training
MAX_LEN          = 128   # tokens per sentence
TRAIN_EPOCHS     = 10
BATCH_SIZE       = 16
LEARNING_RATE    = 2e-5
ANNOTATION_FILE  = "/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/processed/annotation_sample_annotated.csv"  # output of Section 3

# LLM scoring
OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY", "YOUR_KEY_HERE")
LLM_MODEL        = "gpt-4o"
N_REVIEWS_SCORE  = 200   # how many reviews to LLM-score (cost control)

print("Configuration loaded.")

Configuration loaded.


## Sentence Tokenisation

In [6]:
def tokenise_reviews(df: pd.DataFrame) -> pd.DataFrame:
    """
    Explode each review into one row per sentence.
    Returns columns: review_id, sentence_idx, sentence, rating, language_code
    """
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Tokenising"):
        sentences = sent_tokenize(row["review_text"])
        # Drop very short sentences (< 5 words) — often noise
        sentences = [s.strip() for s in sentences if len(s.split()) >= 5]
        for idx, sent in enumerate(sentences):
            rows.append({
                "review_id":    row["review_id"],
                "sentence_idx": idx,
                "sentence":     sent,
                "rating":       row["rating"],
                "language":  row["language_code"],
            })
    return pd.DataFrame(rows)

df_sents = tokenise_reviews(df)
print(f"{len(df_sents):,} sentences from {df['review_id'].nunique():,} reviews")
#df_sents.head(4)

Tokenising:   0%|          | 0/1903442 [00:00<?, ?it/s]

20,275,427 sentences from 1,903,442 reviews


In [23]:
df_sents['language'].unique()
df_sents.columns

Index(['review_id', 'sentence_idx', 'sentence', 'rating', 'language'], dtype='str')

In [ ]:
def detect_language(text):
    try:
        return detect(str(text))
    except:
        return None

##df_sents["detected_lang"] = df_sents["sentence"].apply(detect_language) # takes too long
#non_ascii_mask = df_sents["text"].str.contains(r"[^\x00-\x7F]", na=False) # loo
df_sents["lang"] = df_sents["sentence"].apply(lambda x: classify(str(x))[0])

In [18]:
df_sents[df_sents['review_id']== '948bf3cc5dd00d869c0de5b957c7de2e']
#df_sents[df_sents['review_id']== 'dfdbb7b0eb5a7e4c26d59a937e2e5feb']

,review_id,sentence_idx,sentence,rating,language
13893818,948bf3cc5dd00d869c0de5b957c7de2e,0,3 \n She lifted her face to the stars.,1,eng
13893819,948bf3cc5dd00d869c0de5b957c7de2e,1,"She was Aelin Ashryver Galathynius, heir of tw...",1,eng
13893820,948bf3cc5dd00d869c0de5b957c7de2e,2,She was Aelin Ashryver Galathynius--and she wo...,1,eng
13893821,948bf3cc5dd00d869c0de5b957c7de2e,3,Lo unico bueno que tengo que decir de este lib...,1,eng
13893822,948bf3cc5dd00d869c0de5b957c7de2e,4,En verdad me da mucha pereza escribir esta rev...,1,eng
13893823,948bf3cc5dd00d869c0de5b957c7de2e,5,Seguire la saga porque los libros en si me gus...,1,eng


In [10]:
df_sents.head(25)

,review_id,sentence_idx,sentence,rating
0,dfdbb7b0eb5a7e4c26d59a937e2e5feb,0,This is a special book.,5
1,dfdbb7b0eb5a7e4c26d59a937e2e5feb,1,"It started slow for about the first third, the...",5
2,dfdbb7b0eb5a7e4c26d59a937e2e5feb,2,This is what I love about good science fiction...,5
3,dfdbb7b0eb5a7e4c26d59a937e2e5feb,3,"It is a 2015 Hugo winner, and translated from ...",5
4,dfdbb7b0eb5a7e4c26d59a937e2e5feb,4,For instance the intermixing of Chinese revolu...,5
5,dfdbb7b0eb5a7e4c26d59a937e2e5feb,5,"It is a book about science, and aliens.",5
6,dfdbb7b0eb5a7e4c26d59a937e2e5feb,6,The science described in the book is impressiv...,5
7,dfdbb7b0eb5a7e4c26d59a937e2e5feb,7,Though when it got to folding protons into 8 d...,5
8,dfdbb7b0eb5a7e4c26d59a937e2e5feb,8,But what would happen if our SETI stations rec...,5
9,dfdbb7b0eb5a7e4c26d59a937e2e5feb,9,That part of the book was a bit dark - I would...,5


## Annotation

In [6]:
N_SAMPLE = 3000 

sample = (df_sents.groupby("rating", group_keys=False).apply(lambda g: g.sample(min(len(g), N_SAMPLE // 5), random_state=42)).reset_index(drop=True))
print(f"Sampled {len(sample):,} sentences")

for dim in SENTENCE_DIMS: ## ALL zeros for now
    sample[dim] = 0

sample_path = "/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/processed/annotation_sample.csv"
sample.to_csv(sample_path, index=False)
print(f"Saved to {sample_path}")

Sampled 3,000 sentences
Saved to /user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/processed/annotation_sample.csv


In [17]:
ann_df = pd.read_csv("/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/processed/annotation_sample.csv")
print(ann_df.columns)
tasks = [
    {
        "data": {
            "text": row["sentence"],
            "review_id": row["review_id"],
            "sentence_idx": int(row["sentence_idx"])
        }
    }
    for _, row in ann_df.iterrows()
]

with open("tasks.json", "w") as f:
    json.dump(tasks, f, indent=2)

Index(['review_id', 'sentence_idx', 'sentence', 'language',
       'Narrative Structure & Quality', 'Character & Emotion', 'Originality',
       'Immersion', 'Thematic Depth', 'Writing Style'],
      dtype='str')


In [ ]:
ls = Client(url="http://localhost:8080", api_key="YOUR_API_KEY")

project = ls.start_project(title="Sentence Dimension Detection", label_config=open("config.xml").read())
project.import_tasks("tasks.json")

annotations = project.export_tasks(export_type="JSON")

rows = []
for task in annotations:
    if not task["annotations"]:
        continue
    chosen = set(task["annotations"][0]["result"][0]["value"]["choices"])
    row = {
        "review_id": task["data"]["review_id"],
        "sentence_idx": task["data"]["sentence_idx"],
    }
    for dim in SENTENCE_DIMS:
        row[dim] = 1 if dim in chosen else 0
    rows.append(row)

ann_df = pd.DataFrame(rows)
original_df = pd.read_csv("annotation_sample.csv").drop(columns=SENTENCE_DIMS)
final_df = original_df.merge(ann_df, on=["review_id", "sentence_idx"])
final_df.to_csv("annotation_sample_annotated.csv", index=False)